In [1]:
import torch
import json
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel,Seq2SeqTrainingArguments, Seq2SeqTrainer,default_data_collator
import pandas as pd
import numpy as np
import os
from torch.utils.data import Dataset
import tqdm
from google.colab import drive

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


Run this to check if Donut Works on your machine

In [3]:
print(f"Torch version: {torch.__version__}")

try:
    from transformers import DonutProcessor
    print("DonutProcessor imported successfully!")
except ImportError as e:
    print(f"Import failed: {e}")

Torch version: 2.10.0+cu128
DonutProcessor imported successfully!


Loading data

In [11]:
common_path = os.path.join("/content/drive/MyDrive","Train licenta","Train Dataset")
dataset_path = os.path.join(common_path,"batch_1_images")
label_path = os.path.join(common_path,"batch1_1_updated.csv")
df_labels = pd.read_csv(label_path)
# df_labels.set_index(df_labels['File Name'].map(lambda name: int(name[7:11])).values,inplace=True)


Defining and loading a pretrained model

In [12]:
processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base-finetuned-cord-v2")
model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base-finetuned-cord-v2")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

The image processor of type `DonutImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/536 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/806M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/806M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


VisionEncoderDecoderModel(
  (encoder): DonutSwinModel(
    (embeddings): DonutSwinEmbeddings(
      (patch_embeddings): DonutSwinPatchEmbeddings(
        (projection): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): DonutSwinEncoder(
      (layers): ModuleList(
        (0): DonutSwinStage(
          (blocks): ModuleList(
            (0): DonutSwinLayer(
              (layernorm_before): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (attention): DonutSwinAttention(
                (self): DonutSwinSelfAttention(
                  (query): Linear(in_features=128, out_features=128, bias=True)
                  (key): Linear(in_features=128, out_features=128, bias=True)
                  (value): Linear(in_features=128, out_features=128, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
                )

Proper Donut Parser Class

In [13]:
class DonutInvoiceParser(Dataset):
    def __init__(self,dataset_path,df_labels,processor,max_length=512):
        self.dataset_path = dataset_path
        self.processor = processor
        self.max_length = max_length
        self.df_labels = df_labels

    def __len__(self):
        return len(self.df_labels)

    def __getitem__(self, idx):
        row = self.df_labels.iloc[idx]
        file_name = row['File Name']
        # 1. Load Image
        image_path = os.path.join(self.dataset_path,file_name)
        image = Image.open(image_path).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()

        # 2. Process Labels (Feedback)
        # We turn the JSON metadata into a string the decoder can learn
        json_data = json.loads(row['Json Data'])
        gt_dict = {"gt_parse":json_data}
        gt_string = json.dumps(gt_dict)
        labels = self.processor.tokenizer(
            gt_string,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze()

        # We use -100 to ignore padding in the loss calculation
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}


In [14]:
train_dataset = DonutInvoiceParser(dataset_path=dataset_path,df_labels=df_labels,processor=processor,max_length=512)

In [15]:
new_tokens = ["<invoice_no>", "<total>", "<invoice_date>", "<due_date>", "<qty>"]
processor.tokenizer.add_tokens(new_tokens)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Set essential generation parameters for the model config
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [16]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./donut_invoice_results",
    per_device_train_batch_size=1,     # Keep at 1 for local machines to avoid memory errors
    gradient_accumulation_steps=8,     # This simulates a batch size of 8
    learning_rate=5e-5,                # Standard starting point for fine-tuning
    num_train_epochs=3,                # How many times to see the whole dataset
    logging_steps=10,
    save_steps=100,                    # Save a checkpoint every 100 steps
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),    # Use mixed precision if you have a GPU
    optim="adamw_torch",
    remove_unused_columns=False,
)

In [17]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,       # This is the class we built in the previous step
    data_collator=default_data_collator,
)

In [18]:
trainer.train()

Step,Training Loss
10,53.752588
20,24.541830
30,17.295631
40,14.112843
50,13.175528
60,11.394801
70,10.676128
80,10.076003
90,8.103730
100,8.194862


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=531, training_loss=8.931329351809515, metrics={'train_runtime': 3319.8293, 'train_samples_per_second': 1.277, 'train_steps_per_second': 0.16, 'total_flos': 1.3254978488677171e+19, 'train_loss': 8.931329351809515, 'epoch': 3.0})

In [19]:
trainer.save_model(os.path.join(common_path, "donut_invoice_checkpoint"))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]